# 03 — scGPT Fine-tuning for Perturbation Prediction

Fine-tune scGPT's `TransformerGenerator` on the Adamson dataset to predict post-perturbation gene expression.

**Architecture overview:**
- **Encoder** (pretrained, partially frozen via selective weight loading): converts gene tokens + expression values into contextual embeddings
- **Perturbation head** (new, trained from scratch): modifies embeddings based on which genes are knocked out
- **Decoder** (new): projects back to expression space

**On Mac:** `amp=False`, `use_fast_transformer=False`. Expect ~20-40 min/epoch on CPU. Budget 3-5 hours total.

**Tip:** Run this notebook and leave it running. Progress is logged every 100 batches and to a file. The best checkpoint is saved automatically.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import time
from pathlib import Path

import numpy as np
import torch
from gears import PertData

from src.config import Config
from src.model_utils import load_vocab_and_gene_ids, build_model, load_pretrained_weights
from src.train import run_training

## 1. Configuration

All hyperparameters live in `Config`. The Mac-specific flags are set here.

In [ ]:
config = Config(
    data_dir='../data',
    pretrained_dir='../pretrained/scGPT_human',
    data_name='adamson',
    # Mac-specific flags:
    use_fast_transformer=False,  # no flash-attn without CUDA
    amp=False,                   # GradScaler requires CUDA
    force_cpu=True,              # MPS can't handle seq_len >= 512; use CPU
    # Sequence length: 512 is feasible on CPU (5060 genes → randomly sample 512 per batch)
    # Increase to 1536 on NVIDIA GPU
    max_seq_len=512,
    # Training:
    epochs=15,
    early_stop=10,
    lr=1e-4,
    batch_size=64,
    log_interval=100,
)

print(f'Device: {config.device}')
print(f'max_seq_len: {config.max_seq_len}')
print(f'AMP: {config.amp} | fast_transformer: {config.use_fast_transformer}')

## 2. Load data

In [ ]:
pert_data = PertData(config.data_dir)
pert_data.load(data_name=config.data_name)
pert_data.prepare_split(split=config.split, seed=1)
pert_data.get_dataloader(batch_size=config.batch_size, test_batch_size=config.eval_batch_size)

genes = pert_data.adata.var['gene_name'].tolist()
n_genes = len(genes)
print(f'Genes: {n_genes}')
print(f'Train loader batches: {len(pert_data.dataloader["train_loader"])}')

## 3. Build model

**What happens here:**
1. Load the vocab (60,697 gene tokens)
2. Build a fresh `TransformerGenerator` (12 transformer layers, 512-dim)
3. Load pretrained weights for encoder/value_encoder/transformer_encoder layers only
4. Perturbation head and decoder start from random initialization

In [ ]:
vocab, gene_ids = load_vocab_and_gene_ids(
    config.pretrained_dir, genes, config.special_tokens
)

model = build_model(config, vocab)
n_pretrained_loaded = load_pretrained_weights(
    model, config.pretrained_dir, config.load_param_prefixs
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Gate 1 — Model sanity check

Before starting training, verify the model processes one batch correctly.

In [ ]:
# Run one forward pass to check everything works
batch = next(iter(pert_data.dataloader['train_loader']))
device = config.device
batch_size = len(batch.y)
batch = batch.to(device)
model = model.to(device)

from scgpt.utils import map_raw_id_to_vocab_id
import torch

x = batch.x
ori_gene_values = x[:, 0].view(batch_size, n_genes)
pert_flags = x[:, 1].long().view(batch_size, n_genes)
input_gene_ids = torch.arange(n_genes, device=device, dtype=torch.long)
mapped_ids = map_raw_id_to_vocab_id(input_gene_ids, gene_ids).repeat(batch_size, 1)
src_mask = torch.zeros_like(ori_gene_values, dtype=torch.bool, device=device)

with torch.no_grad():
    out = model(mapped_ids, ori_gene_values, pert_flags,
                src_key_padding_mask=src_mask,
                CLS=False, CCE=False, MVC=False, ECS=False)

print(f'Forward pass OK!')
print(f'  Input shape: {ori_gene_values.shape}')
print(f'  Output shape: {out["mlm_output"].shape}')
assert out['mlm_output'].shape == ori_gene_values.shape
print(f'  Output shape matches input shape — decoder predicts all {n_genes} genes')

# Pretrained weights check
assert n_pretrained_loaded > 0, 'No pretrained weights loaded!'
print(f'\nGate 1 PASSED: model builds and runs forward pass, {n_pretrained_loaded} pretrained tensors loaded')

## 5. Set up logging and save directory

In [ ]:
run_name = f"adamson-{time.strftime('%b%d-%H-%M')}"
save_dir = Path(config.save_dir) / run_name
save_dir.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger(run_name)
logger.setLevel(logging.INFO)

# Console handler
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setFormatter(logging.Formatter('%(asctime)s %(message)s', '%H:%M:%S'))
    logger.addHandler(ch)
    # File handler
    fh = logging.FileHandler(save_dir / 'run.log')
    fh.setFormatter(logging.Formatter('%(asctime)s %(message)s'))
    logger.addHandler(fh)

logger.info(f'Run: {run_name} | Device: {device} | Save: {save_dir}')
print(f'Logging to {save_dir / "run.log"}')

## 6. Train

**On Mac M-series:** Expect ~20-40 minutes per epoch. The first epoch is slowest due to JIT compilation.

Watch for:
- `train_loss` decreasing epoch over epoch ✓
- `val_mse` decreasing and then plateauing ✓
- Early stopping when `val_mse` doesn't improve for 10 epochs

If loss stays flat from epoch 1: check that pretrained weights loaded (n_pretrained_loaded > 0) and that the device is correct.

In [ ]:
# Rebuild model (fresh start, not using the one from Gate 1 check above)
model = build_model(config, vocab).to(device)
load_pretrained_weights(model, config.pretrained_dir, config.load_param_prefixs)

print(f'Starting training...')
print(f'Estimated time on Mac CPU: {config.epochs * 30}–{config.epochs * 40} min max')
print(f'(early stop likely before epoch {config.epochs})')

best_model = run_training(
    model=model,
    pert_data=pert_data,
    config=config,
    gene_ids=gene_ids,
    n_genes=n_genes,
    logger=logger,
    save_dir=save_dir,
)

print(f'\nTraining complete! Best model saved to {save_dir / "best_model.pt"}')

## 7. Gate 3 — Post-training validation

In [ ]:
import os
assert os.path.exists(str(save_dir / 'best_model.pt')), 'Checkpoint not saved!'

# Quick eval on test set
from src.train import eval_perturb
test_results = eval_perturb(
    pert_data.dataloader['test_loader'], best_model, device, config, gene_ids
)

from src.evaluate import compute_delta_pearson
ctrl_adata = pert_data.adata[pert_data.adata.obs['condition'] == 'ctrl']
ctrl_mean = np.asarray(ctrl_adata.X.mean(axis=0)).flatten()

test_metrics = compute_delta_pearson(test_results, ctrl_mean)
print('Test set metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

assert test_metrics.get('pearson_de', 0) > 0.50, \
    f'Model underperformed: pearson_de = {test_metrics["pearson_de"]:.4f}'

print(f'\nGate 3 PASSED. Checkpoint at: {save_dir}')

# Save run path for notebook 04
with open('../outputs/latest_run.txt', 'w') as f:
    f.write(str(save_dir))
print('Run path saved for notebook 04.')